# 04 — Explainability & Error Analysis

Uses SHAP to explain the LightGBM model globally and for individual POs, then slices performance by category/region to check for systematic weaknesses, and shows the resulting buyer/planner decision table.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from po_delay.data_generation import GeneratorParams, generate
from po_delay.decision_rules import build_decision_table
from po_delay.explain import compute_shap_values, explain_instance, global_importance, slice_metrics
from po_delay.features import build_features
from po_delay.models import fit_lightgbm_with_early_stopping
from po_delay.validation import time_based_split

df = generate(GeneratorParams(n_orders=40_000, n_suppliers=150, seed=42))
X, y = build_features(df)
split = time_based_split(df)
X_train, y_train = X.iloc[split.train_idx], y.iloc[split.train_idx]
X_val, y_val = X.iloc[split.val_idx], y.iloc[split.val_idx]
X_test, y_test = X.iloc[split.test_idx], y.iloc[split.test_idx]

model = fit_lightgbm_with_early_stopping(X_train, y_train, X_val, y_val)
proba = model.predict_proba(X_test)[:, 1]

## Global feature importance

In [2]:
shap_values = compute_shap_values(model, X_test)
global_importance(shap_values, list(X_test.columns)).head(10)

,feature,mean_abs_shap
0,supplier_on_time_rate_trailing365,0.289458
1,is_rush_order,0.095466
2,supplier_mean_delay_trailing365,0.046660
3,is_quarter_end,0.043953
4,supplier_open_po_count,0.043078
5,supplier_n_orders_trailing365,0.030378
6,category,0.026259
7,is_holiday_period,0.021481
8,requested_lead_time_days,0.019403
9,supplier_country,0.015018


## A single PO's explanation
The top contributing features for the riskiest PO in the test set — this is the format a buyer-facing tool would show alongside the risk score.

In [3]:
riskiest_idx = int(proba.argmax())
print("Predicted late probability:", proba[riskiest_idx])
explain_instance(shap_values, list(X_test.columns), riskiest_idx)

Predicted late probability: 0.7989835135151567


,feature,shap_value
0,supplier_on_time_rate_trailing365,0.612083
1,is_rush_order,0.431911
2,supplier_mean_delay_trailing365,0.203056
3,is_quarter_end,0.128534
4,is_holiday_period,0.089391


## Error analysis by category and region
Looking for slices where the model underperforms (e.g., small-sample or structurally different segments).

In [4]:
slice_metrics(X_test, y_test.to_numpy(), proba, "category")

,slice,n,low_sample,roc_auc,pr_auc,brier_score,base_rate
0,Raw Materials,1023,False,0.622910,0.279786,0.213283,0.187683
1,Chemicals,1022,False,0.676651,0.404435,0.231973,0.220157
2,Custom Fabrication,997,False,0.650864,0.476338,0.284079,0.336008
3,Electronics,997,False,0.638273,0.328865,0.233182,0.206620
4,MRO,983,False,0.645064,0.294921,0.195410,0.158698
5,Packaging,978,False,0.692249,0.311106,0.208776,0.176892


In [5]:
slice_metrics(X_test, y_test.to_numpy(), proba, "destination_region")

,slice,n,low_sample,roc_auc,pr_auc,brier_score,base_rate
0,EU-East,1061,False,0.658020,0.327411,0.232693,0.194156
1,APAC,1017,False,0.682266,0.417540,0.225873,0.214356
2,NA-West,996,False,0.693394,0.401765,0.225703,0.216867
3,NA-East,991,False,0.701615,0.396263,0.221885,0.216953
4,EU-West,979,False,0.675110,0.353979,0.228087,0.214505
5,LATAM,956,False,0.662573,0.370591,0.232907,0.232218


## Buyer/planner decision table
Translating raw probabilities into a risk tier and a recommended action (see `docs/model_card.md` for the full tier definitions and their caveats).

In [6]:
build_decision_table(proba).head(10)

,risk_probability,risk_tier,recommended_action
0,0.640195,Critical,Escalate: evaluate alternate sourcing / expedi...
1,0.495409,High,Buyer follow-up call with supplier; evaluate e...
2,0.542902,High,Buyer follow-up call with supplier; evaluate e...
3,0.473927,High,Buyer follow-up call with supplier; evaluate e...
4,0.448058,High,Buyer follow-up call with supplier; evaluate e...
5,0.379452,High,Buyer follow-up call with supplier; evaluate e...
6,0.295670,Medium,Automated supplier check-in email.
7,0.422823,High,Buyer follow-up call with supplier; evaluate e...
8,0.570742,High,Buyer follow-up call with supplier; evaluate e...
9,0.427891,High,Buyer follow-up call with supplier; evaluate e...
